# Assignment 02, Part 2: `word_count_batch_submit.py`
**Name:** Devi Aditya Dantuluri

Batch word counting over all the text files in the `gutenberg_books` folder with one PySpark routine. Questions 05 and 06.

## Setup: download every book in the `gutenberg_books` repository

The course data repository (`jonesberg/DataAnalysisWithPythonAndPySpark-Data`, branch `trunk`) is used. The books are downloaded with the same `urllib` loader as before, into a local `gutenberg_books/` folder.

In [1]:
import os, glob, shutil, time
import urllib.request

repo = "https://raw.githubusercontent.com/jonesberg/DataAnalysisWithPythonAndPySpark-Data/trunk/gutenberg_books/"
book_files = ["11-0.txt", "1342-0.txt", "1661-0.txt", "2701-0.txt", "30254-0.txt", "84-0.txt"]

os.makedirs("gutenberg_books", exist_ok=True)
for name in book_files:
    urllib.request.urlretrieve(repo + name, os.path.join("gutenberg_books", name))

# 7th file: War and Peace by Leo Tolstoy, Project Gutenberg eBook #2600 (https://www.gutenberg.org/ebooks/2600).
wp_path = os.path.join("gutenberg_books", "2600-0.txt")
try:
    urllib.request.urlretrieve("https://www.gutenberg.org/files/2600/2600-0.txt", wp_path)
    print("War and Peace downloaded from gutenberg.org")
except Exception as e:
    # Fallback: the same Gutenberg text mirrored on GitHub (GITenberg project).
    urllib.request.urlretrieve("https://raw.githubusercontent.com/GITenberg/War-and-Peace_2600/master/2600-0.txt", wp_path)
    print("gutenberg.org unreachable; War and Peace downloaded from the GITenberg mirror")

# Print the title found in each file's Gutenberg header, and the file size.
for path in sorted(glob.glob("gutenberg_books/*.txt")):
    title = next((l.strip() for l in open(path, encoding="utf-8") if l.startswith("Title:")), "?")
    print(f"{os.path.basename(path):12s} {os.path.getsize(path):>9,d} bytes   {title}")
print("\nNumber of text files:", len(glob.glob("gutenberg_books/*.txt")))

gutenberg.org unreachable; War and Peace downloaded from the GITenberg mirror
11-0.txt       169,859 bytes   Title: Alice’s Adventures in Wonderland
1342-0.txt     711,299 bytes   Title: Pride and Prejudice
1661-0.txt     595,478 bytes   Title: The Adventures of Sherlock Holmes
2600-0.txt   3,359,542 bytes   Title: War and Peace
2701-0.txt   1,253,868 bytes   Title: Moby Dick; or The Whale
30254-0.txt  1,057,984 bytes   Title: The Romance of Lust A classic Victorian erotic novel
84-0.txt       442,949 bytes   Title: Frankenstein

Number of text files: 7


**Note:** the `gutenberg_books` folder in the course repository currently contains six books (`11-0`, `1342-0`, `1661-0`, `2701-0`, `30254-0`, `84-0`). To reach the seven files mentioned in the assignment, *War and Peace* by Leo Tolstoy (Project Gutenberg #2600) was added as `2600-0.txt`. It is by far the largest file (about 3.2 MB, roughly 4.5 times the size of *Pride and Prejudice*), so it dominates the batch counts. The code works unchanged for any number of files because the wildcard `*.txt` is used.

In [2]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = (SparkSession.builder
         .appName("Counting word occurences from a book.")
         .config("spark.sql.adaptive.enabled", "false")   # same setting as Part 1, so partition counts are comparable
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

## The batch program

All operations are chained into one expression. The only change from the original script is the input path: `gutenberg_books/*.txt` instead of `gutenberg_books/1342-0.txt`. Spark expands the wildcard and reads every matching file as if it were one big text file (one row per line from every file).

In [3]:
def word_counts(path):
    """The word_count_batch_submit.py chain, parameterised by the input path."""
    return (
        spark.read.text(path)
        .select(F.split(F.col("value"), " ").alias("line"))
        .select(F.explode(F.col("line")).alias("word"))
        .select(F.lower(F.col("word")).alias("word"))
        .select(F.regexp_extract(F.col("word"), "[a-z']*", 0).alias("word"))
        .where(F.col("word") != "")
        .groupby(F.col("word"))
        .count()
    )

# Single novel (for comparison) and the whole batch.
results_novel = word_counts("gutenberg_books/1342-0.txt")
results_batch = word_counts("gutenberg_books/*.txt")

print("Input partitions, single novel:", spark.read.text("gutenberg_books/1342-0.txt").rdd.getNumPartitions())
print("Input partitions, batch       :", spark.read.text("gutenberg_books/*.txt").rdd.getNumPartitions())

Input partitions, single novel: 1
Input partitions, batch       : 1


In [4]:
# Write the batch results to a single-partition file, exactly as in the script.
shutil.rmtree("results_single_partition.csv", ignore_errors=True)
results_batch.coalesce(1).write.csv("results_single_partition.csv")
print(os.listdir("results_single_partition.csv"))

['._SUCCESS.crc', '_SUCCESS', 'part-00000-b2e8ec2d-5634-411f-8023-524082046a53-c000.csv', '.part-00000-b2e8ec2d-5634-411f-8023-524082046a53-c000.csv.crc']


## Question 05: does the single-partition file contain results from all files?

In [5]:
# Read the written file back and compare it with the counts computed directly from the DataFrames.
written = spark.read.csv("results_single_partition.csv").toDF("word", "count").withColumn("count", F.col("count").cast("long"))

lines_novel = spark.read.text("gutenberg_books/1342-0.txt").count()
lines_batch = spark.read.text("gutenberg_books/*.txt").count()
lines_sum   = sum(spark.read.text(p).count() for p in sorted(glob.glob("gutenberg_books/*.txt")))

print("Lines read, single novel :", lines_novel)
print("Lines read, batch (*.txt):", lines_batch, "| sum over the individual files:", lines_sum)
print()
print("Distinct words, novel   :", results_novel.count())
print("Distinct words, batch   :", results_batch.count())
print("Rows in written CSV file:", written.count())
print()
print("Total word occurrences, novel:", results_novel.agg(F.sum("count")).first()[0])
print("Total word occurrences, batch:", results_batch.agg(F.sum("count")).first()[0])
print("Total word occurrences, file :", written.agg(F.sum("count")).first()[0])

Lines read, single novel : 13427
Lines read, batch (*.txt): 143965 | sum over the individual files: 143965



Distinct words, novel   : 6595


Distinct words, batch   : 29042
Rows in written CSV file: 29042



Total word occurrences, novel: 122175


Total word occurrences, batch: 1291358


Total word occurrences, file : 1291358


**Confirmation.** Yes. The number of lines read with `*.txt` equals the sum of the lines of the individual files, the batch result has far more distinct words than the novel alone, and the single-partition CSV has exactly the same number of rows and the same total count as the batch DataFrame. `coalesce(1)` only changes how the output is laid out on disk (one file instead of many); it does not drop any data, so the file holds the counts and, after sorting, the ranking for all the books together.

In [6]:
# Direct evidence: words that appear in the batch file but not in Pride and Prejudice at all.
# These can only come from the other books (Moby Dick, Frankenstein, Alice, Sherlock Holmes, War and Peace, ...).
probe = ["whale", "ahab", "holmes", "watson", "alice", "hatter", "frankenstein", "monster", "darcy", "elizabeth",
         "pierre", "natásha", "napoleon", "moscow", "kutúzov", "prince"]
cmp = (results_novel.withColumnRenamed("count", "novel_count")
       .join(results_batch.withColumnRenamed("count", "batch_count"), "word", "full")
       .join(written.withColumnRenamed("count", "file_count"), "word", "full")
       .where(F.col("word").isin(probe))
       .fillna(0)
       .orderBy(F.col("batch_count").desc()))
cmp.show(truncate=False)

+------------+-----------+-----------+----------+
|word        |novel_count|batch_count|file_count|
+------------+-----------+-----------+----------+
|pierre      |0          |1946       |1946      |
|prince      |0          |1906       |1906      |
|whale       |0          |1235       |1235      |
|moscow      |0          |716        |716       |
|elizabeth   |594        |685        |685       |
|napoleon    |0          |576        |576       |
|ahab        |0          |504        |504       |
|holmes      |0          |460        |460       |
|alice       |0          |409        |409       |
|darcy       |374        |374        |374       |
|monster     |0          |85         |85        |
|watson      |0          |80         |80        |
|hatter      |0          |56         |56        |
|frankenstein|0          |31         |31        |
+------------+-----------+-----------+----------+



### Differences between batch processing and the single novel

Top-10 lists side by side, and the words whose rank changes most.

In [7]:
print("Top 10, single novel (Pride and Prejudice):")
results_novel.orderBy("count", ascending=False).show(10)
print("Top 10, batch of all books (from the single-partition file):")
written.orderBy("count", ascending=False).show(10)

Top 10, single novel (Pride and Prejudice):


+----+-----+
|word|count|
+----+-----+
| the| 4480|
|  to| 4218|
|  of| 3711|
| and| 3504|
| her| 2199|
|   a| 1982|
|  in| 1909|
| was| 1838|
|   i| 1749|
| she| 1668|
+----+-----+
only showing top 10 rows
Top 10, batch of all books (from the single-partition file):


+----+-----+
|word|count|
+----+-----+
| the|73153|
| and|45737|
|  to|37156|
|  of|36162|
|   a|24864|
|  in|21658|
|   i|17577|
|that|17525|
| was|16243|
|  he|16152|
+----+-----+
only showing top 10 rows


In [8]:
# Same word, different counts: the top words are the same stop words, but the counts are several times larger
# in the batch, and the order changes (e.g. "the"/"of"/"and" dominate the batch while "to"/"her" fall back:
# Pride and Prejudice is dialogue-heavy with a female protagonist, Moby Dick is descriptive prose).
top_cmp = (results_novel.withColumnRenamed("count", "novel_count")
           .join(results_batch.withColumnRenamed("count", "batch_count"), "word", "outer")
           .fillna(0)
           .withColumn("ratio_batch_over_novel", F.round(F.col("batch_count") / F.greatest(F.col("novel_count"), F.lit(1)), 1)))
top_cmp.orderBy(F.col("batch_count").desc()).show(20)

+----+-----------+-----------+----------------------+
|word|novel_count|batch_count|ratio_batch_over_novel|
+----+-----------+-----------+----------------------+
| the|       4480|      73153|                  16.3|
| and|       3504|      45737|                  13.1|
|  to|       4218|      37156|                   8.8|
|  of|       3711|      36162|                   9.7|
|   a|       1982|      24864|                  12.5|
|  in|       1909|      21658|                  11.3|
|   i|       1749|      17577|                  10.0|
|that|       1487|      17525|                  11.8|
| was|       1838|      16243|                   8.8|
|  he|       1296|      16152|                  12.5|
| his|       1247|      14596|                  11.7|
|  it|       1482|      14422|                   9.7|
| her|       2199|      12636|                   5.7|
|with|       1092|      12251|                  11.2|
| had|       1170|      11069|                   9.5|
|  as|       1174|      1035

In [9]:
# Informative words after removing the same stop-word list as Part 1: the batch top-20 is a mixture of
# characters/themes from all books instead of Austen's characters only.
stop_words = [
    "the", "a", "an", "this", "that", "these", "those", "some", "any", "every", "each", "no",
    "i", "me", "my", "mine", "myself", "you", "your", "yours", "yourself", "he", "him", "his", "himself",
    "she", "her", "hers", "herself", "it", "its", "itself", "we", "us", "our", "ours", "they", "them",
    "their", "theirs", "themselves", "who", "whom", "whose", "which", "what", "one", "all", "both",
    "am", "is", "are", "was", "were", "be", "been", "being", "have", "has", "had", "having",
    "do", "does", "did", "will", "would", "shall", "should", "can", "could", "may", "might", "must",
    "of", "to", "in", "on", "at", "by", "for", "with", "from", "into", "upon", "about", "as",
    "than", "over", "after", "before", "between", "through", "without", "under",
    "and", "but", "or", "nor", "so", "if", "then", "there", "here", "when", "where", "how", "why",
    "not", "very", "such", "much", "more", "most", "own", "only", "too", "also", "again",
    "out", "up", "down", "now", "ever", "never", "yet", "still", "though", "however", "well",
    "s", "t", "'s", "'",
]
print("Top 20 informative words, single novel:")
results_novel.where(~F.col("word").isin(stop_words)).orderBy("count", ascending=False).show(20)
print("Top 20 informative words, batch of all books:")
results_batch.where(~F.col("word").isin(stop_words)).orderBy("count", ascending=False).show(20)

Top 20 informative words, single novel:


+---------+-----+
|     word|count|
+---------+-----+
|       mr|  766|
|elizabeth|  594|
|     said|  401|
|    darcy|  374|
|      mrs|  338|
|   bennet|  294|
|     miss|  272|
|     jane|  266|
|  bingley|  255|
|     know|  237|
|    other|  224|
|     soon|  216|
|    think|  211|
|     time|  200|
|     good|  190|
|   little|  188|
|   sister|  180|
|     lady|  176|
|  nothing|  173|
|     make|  168|
+---------+-----+
only showing top 20 rows
Top 20 informative words, batch of all books:


+-------+-----+
|   word|count|
+-------+-----+
|   said| 4933|
|    man| 2377|
|   time| 2239|
| pierre| 1946|
| prince| 1906|
|  other| 1844|
|    see| 1772|
|   like| 1763|
| little| 1758|
|   know| 1691|
|  first| 1652|
|    two| 1555|
|    old| 1492|
|thought| 1396|
|   long| 1377|
|   eyes| 1372|
|    way| 1335|
|   went| 1318|
|   room| 1309|
|     go| 1303|
+-------+-----+
only showing top 20 rows


**What changes with batch processing.**
* The vocabulary grows from 6,595 distinct words to 29,042 (about 4.4x), and total occurrences grow from 122,175 to 1,291,358 (about 10.6x).
* Words absent from *Pride and Prejudice* (`pierre`, `prince`, `whale`, `moscow`, `napoleon`, `ahab`, `holmes`, `alice`, ...) appear with high counts; they come from the other files, which confirms the batch read.
* The stop words stay at the top but their relative order shifts (`and` overtakes `to` and `of`; `he` and `his` enter the top 10 while `her` and `she` fall out, because *War and Peace* and *Moby Dick*, the two largest books, are centred on male characters).
* After stop-word removal the novel's top words are Austen's characters (`mr`, `elizabeth`, `darcy`, `bennet`), while the batch top words are dominated by the largest books (`pierre` and `prince` from *War and Peace* enter the top 5, and generic narrative words such as `man`, `see`, `like`, `old`, `eyes`, `room` take over), so the single-file ranking is no longer representative of any one novel.

## Question 06: computational cost analysis for batch processing

Same benchmark as Question 04 (three runs per variant, mean reported), now with `gutenberg_books/*.txt` as the input.

In [10]:
def run_pipeline(path, out_dir, n_coalesce=None):
    shutil.rmtree(out_dir, ignore_errors=True)
    t0 = time.perf_counter()
    res = word_counts(path)
    top10 = res.orderBy("count", ascending=False).take(10)
    if n_coalesce is None:
        res.write.csv(out_dir)
    else:
        res.coalesce(n_coalesce).write.csv(out_dir)
    return time.perf_counter() - t0

def benchmark(path, label_prefix, repeats=3):
    import pandas as pd
    variants = [("without coalesce", None), ("coalesce(1)", 1), ("coalesce(4)", 4)]
    rows = []
    for name, n in variants:
        times = [run_pipeline(path, f"timing_{label_prefix}_{n}_{r}.csv", n) for r in range(repeats)]
        rows.append({"variant": name, "run_1_s": times[0], "run_2_s": times[1], "run_3_s": times[2],
                     "mean_s": sum(times) / len(times), "mean_min": sum(times) / len(times) / 60,
                     "files_written": len(glob.glob(f"timing_{label_prefix}_{n}_0.csv/part-*"))})
    return pd.DataFrame(rows).round(3)

print("Single novel (repeat of Question 04 in this session, for a like-for-like comparison):")
timing_novel = benchmark("gutenberg_books/1342-0.txt", "novel")
display(timing_novel)

print("Batch of all books:")
timing_batch = benchmark("gutenberg_books/*.txt", "batch")
display(timing_batch)

Single novel (repeat of Question 04 in this session, for a like-for-like comparison):


,variant,run_1_s,run_2_s,run_3_s,mean_s,mean_min,files_written
0,without coalesce,8.547,7.203,6.843,7.531,0.126,200
1,coalesce(1),1.355,1.507,1.201,1.354,0.023,1
2,coalesce(4),1.522,1.470,1.625,1.539,0.026,4


Batch of all books:


,variant,run_1_s,run_2_s,run_3_s,mean_s,mean_min,files_written
0,without coalesce,7.767,8.445,7.934,8.049,0.134,200
1,coalesce(1),2.669,2.802,2.874,2.782,0.046,1
2,coalesce(4),2.929,2.989,2.633,2.850,0.048,4


In [11]:
# Side-by-side summary.
import pandas as pd
summary = pd.DataFrame({
    "variant": timing_novel["variant"],
    "novel_mean_s": timing_novel["mean_s"],
    "batch_mean_s": timing_batch["mean_s"],
})
summary["batch / novel"] = (summary["batch_mean_s"] / summary["novel_mean_s"]).round(2)
summary

,variant,novel_mean_s,batch_mean_s,batch / novel
0,without coalesce,7.531,8.049,1.07
1,coalesce(1),1.354,2.782,2.05
2,coalesce(4),1.539,2.850,1.85


**Interpretation.** The batch input is about 10.6 times larger (7.6 MB vs 0.7 MB, 143,965 lines vs 13,427), yet Spark still reads it as a single input partition because all seven files together are far below the default 128 MB partition size. The read/split/explode/count stages and the shuffle therefore do more work on one core, and the `coalesce(1)` and `coalesce(4)` runs grow from about 1.4 to 1.5 s to about 2.8 s (roughly 2x), far less than 10x, because a large share of each run is fixed Spark overhead (job scheduling, task launch, file commit) that does not depend on data size.

Without `coalesce` the batch takes about 8 s versus 7.5 s for the single novel: that variant is dominated by launching and committing 200 write tasks, which costs the same regardless of how much data is in each file. The ranking of the variants is the same in both cases: no coalesce is slowest; `coalesce(1)` and `coalesce(4)` are close, with `coalesce(1)` marginally faster on this small data because a single writer avoids extra task overhead.

In [12]:
for d in glob.glob("timing_*_*.csv"):
    shutil.rmtree(d, ignore_errors=True)
spark.stop()